In [83]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/My Drive/AIML-Project/G6/weatherHistory.csv")
print("Shape:", df.shape)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (96453, 12)


,Formatted Date,Summary,Precip Type,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Loud Cover,Pressure (millibars),Daily Summary
0,2006-04-01 00:00:00.000 +0200,Partly Cloudy,rain,9.472222,7.388889,0.89,14.1197,251.0,15.8263,0.0,1015.13,Partly cloudy throughout the day.
1,2006-04-01 01:00:00.000 +0200,Partly Cloudy,rain,9.355556,7.227778,0.86,14.2646,259.0,15.8263,0.0,1015.63,Partly cloudy throughout the day.
2,2006-04-01 02:00:00.000 +0200,Mostly Cloudy,rain,9.377778,9.377778,0.89,3.9284,204.0,14.9569,0.0,1015.94,Partly cloudy throughout the day.
3,2006-04-01 03:00:00.000 +0200,Partly Cloudy,rain,8.288889,5.944444,0.83,14.1036,269.0,15.8263,0.0,1016.41,Partly cloudy throughout the day.
4,2006-04-01 04:00:00.000 +0200,Mostly Cloudy,rain,8.755556,6.977778,0.83,11.0446,259.0,15.8263,0.0,1016.51,Partly cloudy throughout the day.


In [84]:
df.isnull().sum()

,0
Formatted Date,0
Summary,0
Precip Type,517
Temperature (C),0
Apparent Temperature (C),0
Humidity,0
Wind Speed (km/h),0
Wind Bearing (degrees),0
Visibility (km),0
Loud Cover,0


In [85]:
duplicates = df[df.duplicated()]
duplicates.shape

(24, 12)

In [86]:
df['Precip Type'] = df['Precip Type'].ffill() #ffill(), bfill(), mode() gives same values(assign to most rain)

In [87]:
df.isnull().sum()

,0
Formatted Date,0
Summary,0
Precip Type,0
Temperature (C),0
Apparent Temperature (C),0
Humidity,0
Wind Speed (km/h),0
Wind Bearing (degrees),0
Visibility (km),0
Loud Cover,0


In [88]:
df['Precip Type'].value_counts()

,count
Precip Type,
rain,85741
snow,10712


In [89]:
df['Summary'].value_counts()

,count
Summary,
Partly Cloudy,31733
Mostly Cloudy,28094
Overcast,16597
Clear,10890
Foggy,7148
Breezy and Overcast,528
Breezy and Mostly Cloudy,516
Breezy and Partly Cloudy,386
Dry and Partly Cloudy,86


In [90]:
df['Suitability'] = (df['Summary'] == 'Clear').astype(int)

In [91]:
df = df.drop_duplicates()

In [92]:
df = pd.get_dummies(
    df,
    columns=['Precip Type'],
    prefix='Precip',
    dtype=int
)

Handle Loud Cover

In [93]:
df['Loud Cover'].value_counts()

,count
Loud Cover,
0.0,96429


In [94]:
df = df.drop(columns=['Formatted Date', 'Summary', 'Daily Summary', 'Loud Cover'])

In [95]:
df.head()

,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Pressure (millibars),Suitability,Precip_rain,Precip_snow
0,9.472222,7.388889,0.89,14.1197,251.0,15.8263,1015.13,0,1,0
1,9.355556,7.227778,0.86,14.2646,259.0,15.8263,1015.63,0,1,0
2,9.377778,9.377778,0.89,3.9284,204.0,14.9569,1015.94,0,1,0
3,8.288889,5.944444,0.83,14.1036,269.0,15.8263,1016.41,0,1,0
4,8.755556,6.977778,0.83,11.0446,259.0,15.8263,1016.51,0,1,0


In [96]:
df = df.rename(columns={
    'Temperature (C)': 'Temperature',
    'Apparent Temperature (C)': 'Apparent_Temperature',
    'Wind Speed (km/h)': 'Wind_Speed',
    'Wind Bearing (degrees)': 'Wind_Bearing',
    'Visibility (km)': 'Visibility',
    'Pressure (millibars)': 'Pressure',
    'Precip_rain': 'Precip_Rain',
    'Precip_snow': 'Precip_Snow'
})

Scaling the Dataset

In [97]:
from sklearn.preprocessing import StandardScaler

num_cols = ['Temperature', 'Apparent_Temperature', 'Humidity', 'Wind_Speed', 'Wind_Bearing', 'Visibility', 'Pressure']
scaler = StandardScaler()
scaler.fit(df[num_cols])

StandardScaler()

In [98]:
df.head()

,Temperature,Apparent_Temperature,Humidity,Wind_Speed,Wind_Bearing,Visibility,Pressure,Suitability,Precip_Rain,Precip_Snow
0,9.472222,7.388889,0.89,14.1197,251.0,15.8263,1015.13,0,1,0
1,9.355556,7.227778,0.86,14.2646,259.0,15.8263,1015.63,0,1,0
2,9.377778,9.377778,0.89,3.9284,204.0,14.9569,1015.94,0,1,0
3,8.288889,5.944444,0.83,14.1036,269.0,15.8263,1016.41,0,1,0
4,8.755556,6.977778,0.83,11.0446,259.0,15.8263,1016.51,0,1,0


In [99]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import mean_squared_error, r2_score

In [100]:
X = df.drop(columns=['Suitability'])
y = df['Suitability']

pd_X = pd.DataFrame(X)
pd_y = pd.Series(y)

pd_X.head()

,Temperature,Apparent_Temperature,Humidity,Wind_Speed,Wind_Bearing,Visibility,Pressure,Precip_Rain,Precip_Snow
0,9.472222,7.388889,0.89,14.1197,251.0,15.8263,1015.13,1,0
1,9.355556,7.227778,0.86,14.2646,259.0,15.8263,1015.63,1,0
2,9.377778,9.377778,0.89,3.9284,204.0,14.9569,1015.94,1,0
3,8.288889,5.944444,0.83,14.1036,269.0,15.8263,1016.41,1,0
4,8.755556,6.977778,0.83,11.0446,259.0,15.8263,1016.51,1,0


In [101]:
pd_y.head()

,Suitability
0,0
1,0
2,0
3,0
4,0


In [102]:
X_train, X_test, y_train, y_test = train_test_split(pd_X, pd_y, test_size=0.2, random_state=42)
print(f"Training Samples: {len(X_train)}, Testing Samples: {len(X_test)}")

Training Samples: 77143, Testing Samples: 19286


In [ ]:
for kernel in ['linear', 'rbf']:
  model_svm = SVC(kernel=kernel, probability=True, random_state=42)
  model_svm.fit(X_train, y_train)
  print(f"\n SVM ({kernel}) Support Vectors:", len(model_svm. support_vectors_))

In [ ]:
for kernel in ['linear', 'rbf']:
  model_svm = SVC(kernel=kernel, probability=True, random_state=42).fit(X_train, y_train)
  y_pred_svm = model_svm.predict(X_test)
  print(f"\nSVM ({kernel}) Metrics:")
  print(f"Accuracy: {accuracy_score(y_test, y_pred_svm) :. 2f}")
  print(f"Precision: {precision_score(y_test, y_pred_svm) :. 2f}")
  print(f"Recall: {recall_score(y_test, y_pred_svm) :. 2f}")
  print(f"F1 Score: {f1_score(y_test, y_pred_svm) :. 2f}")
  print(f"ROC AUC: {roc_auc_score(y_test, model_svm.predict_proba(X_test)[:, 1]) :. 2f}")

  cm = confusion_matrix(y_test, y_pred_svm)
  disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)
  disp.plot(cmap=plt.cm.plasma)
  plt.title("Confusion Matrix")
  plt.show()

Mean Squared Error (MSE): 0.09
Root Mean Squared Error (RMSE): 0.31
R2 Score: 0.06


In [ ]:
pca = PCA(n_components=2)
X_train_pca = pca. fit_transform(X_train)
X_test_pca = pca.transform(X_test)

In [ ]:
for kernel in ['linear', 'rbf']:

  model_svm_pca = SVC(kernel=kernel).fit(X_train_pca, y_train)

  x_min, x_max = X_test_pca[:,0].min() - 1, X_test_pca[:, 0].max() + 1
  y_min, y_max = X_test_pca[:, 1].min() - 1, X_test_pca[:, 1].max() + 1
  xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))
  Z = model_svm_pca.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

  plt.figure(figsize=(6, 4))
  plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
  plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, cmap='coolwarm', edgecolors='k')
  plt.title(f"SVM Decision Boundary ({kernel}, PCA)")
  plt.xlabel("PCA Component 1")
  plt.ylabel("PCA Component 2")
  plt.show()

In [ ]:
for C in [0.1, 1, 10]:
  model_svm = SVC(kernel='rbf', C=C, probability=True, random_state=42). fit(X_train, y_train)
  y_pred_svm = model_svm.predict(X_test)
  print(f"SVM (RBF, C={C}) Metrics:")
  print(f"Accuracy: {accuracy_score(y_test, y_pred_svm) :. 2f}, F1: {f1_score(y_test, y_pred_svm) :. 2f}")